# 04. Three-dimensional geological model and individual units

This notebook plots the full geological model and three individual geological units. The full-model view uses the global exterior faces and overlays the initial triangle net on the Ground interface. Individual-unit views use each unit's own boundary faces.

**Input:** `demo_model.npz` and `completed_drill_surfaces.csv`, generated by `run_model.py`. The input directory is selected by `HG3D_MODEL_DIR` (default: `outputs/model/`).

**Output:** `3D geological model.png`, `lacustrine.png`, `deltaic.png`, and `fluvial.png`. Figures are displayed and saved to `HG3D_FIGURE_DIR` (default: `outputs/figures/`).

**Interpretation:** The full-model figure is 12.5 × 9.2 cm; each individual-unit figure is 4.5 × 4 cm. The vertical display scale is 2.5. Lighting and display offsets affect the rendering only.


## 1. Imports and configuration


In [ ]:
import csv
import os
import tempfile
from collections import defaultdict
from pathlib import Path

# Matplotlib needs a writable cache directory before its first import.
MPLCONFIG_DIR = Path(tempfile.gettempdir()) / "matplotlib-harmonicgeo3d"
MPLCONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIG_DIR))

# Project paths.
PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "run_model.py").is_file() and (path / "notebooks").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Start this notebook from the HarmonicGeo3D directory or a subdirectory."
    )

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import to_rgb, to_rgba
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator
from mpl_toolkits.mplot3d.art3d import Line3DCollection, Poly3DCollection

# Input data.
MODEL_DATA_DIR = Path(
    os.environ.get("HG3D_MODEL_DIR", PROJECT_ROOT / "outputs" / "model")
)
MODEL_PATH = MODEL_DATA_DIR / "demo_model.npz"
COMPLETED_BOREHOLE_PATH = MODEL_DATA_DIR / "completed_drill_surfaces.csv"

# Unit labels and colours follow body_names from bottom to top.
BODY_LABELS = ["Coal", "Lacustrine", "Deltaic", "Fluvial", "Topsoil"]
BODY_COLORS = ["#41464B", "#849EAF", "#88AAA3", "#C7AE80", "#B8B0A3"]

# Camera elevation, azimuth, and roll.
VIEW_ELEV, VIEW_AZIM, VIEW_ROLL = 30, -105, 0
BOREHOLE_LABEL_SIZE = 5
# Diffuse lighting changes display colours without changing model geometry.
LIGHT_AZIMUTH = 315
LIGHT_ALTITUDE = 45
AMBIENT_LIGHT = 0.65
DIFFUSE_LIGHT = 0.30
LIGHT_RELIEF_SCALE = 1.3
Z_DISPLAY_SCALE = 2.5
Z_TICK_SPACING_M = 40
GROUND_MESH_OFFSET = 0.25

FULL_FIGURE_WIDTH_CM, FULL_FIGURE_HEIGHT_CM = 12.5, 9.2
UNIT_FIGURE_WIDTH_CM, UNIT_FIGURE_HEIGHT_CM = 4.5, 4.0
UNIT_PANELS = (
    ("body-01", "Lacustrine", "lacustrine"),
    ("body-02", "Deltaic", "deltaic"),
    ("body-03", "Fluvial", "fluvial"),
)

CM_PER_INCH = 2.54
FONT_FAMILY = ["Arial", "DejaVu Sans"]
PLOT_RC = {
    "font.family": FONT_FAMILY,
    "axes.unicode_minus": False,
    "font.size": 9,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
    "figure.dpi": 120,
}

# Figure output.
FIGURE_DIR = Path(
    os.environ.get("HG3D_FIGURE_DIR", PROJECT_ROOT / "outputs" / "figures")
)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## 2. Data loading and validation


In [ ]:
with np.load(MODEL_PATH, allow_pickle=False) as model_data:
    body_names = model_data["body_names"].astype(str)
    initial_nodes_xy = np.asarray(model_data["initial_nodes_xy"], dtype=float)
    initial_triangles = np.asarray(model_data["initial_triangles"], dtype=int)
    final_nodes_xy = np.asarray(model_data["final_nodes_xy"], dtype=float)
    final_triangles = np.asarray(model_data["final_triangles"], dtype=int)
    interfaces_z = np.asarray(model_data["Z"], dtype=float)
    cell_presence_mask = np.asarray(model_data["cell_presence_mask"], dtype=bool)
    vertices_xyz = np.asarray(model_data["vertices_xyz"], dtype=float)
    tetrahedra = np.asarray(model_data["tetrahedra"], dtype=int)
    tetra_body_ids = np.asarray(model_data["tetra_body_ids"], dtype=int)
    original_node_count = int(np.asarray(model_data["original_node_count"]))

n_interfaces = interfaces_z.shape[1]
n_bodies = n_interfaces - 1
if final_nodes_xy.shape[0] != interfaces_z.shape[0]:
    raise ValueError("final_nodes_xy and Z must have the same node count.")
if final_triangles.ndim != 2 or final_triangles.shape[1] != 3:
    raise ValueError("final_triangles must have shape (n_triangle, 3).")
if cell_presence_mask.shape != (final_triangles.shape[0], n_bodies):
    raise ValueError(
        "cell_presence_mask must match the triangle and geological-unit counts."
    )
if len(BODY_LABELS) != n_bodies or len(BODY_COLORS) != n_bodies:
    raise ValueError(
        f"The legend configuration must match the {n_bodies} geological units in the model."
    )
if initial_nodes_xy.shape[0] != original_node_count:
    raise ValueError("The initial-node count must match original_node_count.")
if not np.allclose(initial_nodes_xy, final_nodes_xy[:original_node_count]):
    raise ValueError(
        "The first original_node_count rows of final_nodes_xy must match initial_nodes_xy."
    )

with COMPLETED_BOREHOLE_PATH.open("r", encoding="utf-8-sig", newline="") as stream:
    completed_boreholes = [
        {
            "drill": row["drill"],
            "x": float(row["coord_x"]),
            "y": float(row["coord_y"]),
            "ground": float(row["Ground"]),
        }
        for row in csv.DictReader(stream)
    ]

## 3. Boundary-face preparation and lighting


In [ ]:
x_origin_m, y_origin_m = final_nodes_xy.min(axis=0)
x_span_m, y_span_m = final_nodes_xy.max(axis=0) - final_nodes_xy.min(axis=0)
z_min_m, z_max_m = (
    float(np.nanmin(vertices_xyz[:, 2])),
    float(np.nanmax(vertices_xyz[:, 2])),
)

# Individual-unit views include the boundary faces of each geological unit.
body_faces = []
for body_index in range(n_bodies):
    counts, oriented = defaultdict(int), {}
    for tet in tetrahedra[tetra_body_ids == body_index]:
        a, b, c, d = map(int, tet)
        for face in ((a, b, c), (a, b, d), (a, c, d), (b, c, d)):
            key = tuple(sorted(face))
            counts[key] += 1
            oriented[key] = face
    body_faces.append(
        [
            vertices_xyz[list(oriented[key])]
            for key, count in counts.items()
            if count == 1
        ]
    )

# The full-model view excludes shared internal faces to avoid visual occlusion.
counts, oriented, owners = defaultdict(int), {}, {}
for tet, body_index in zip(tetrahedra, tetra_body_ids):
    a, b, c, d = map(int, tet)
    for face in ((a, b, c), (a, b, d), (a, c, d), (b, c, d)):
        key = tuple(sorted(face))
        counts[key] += 1
        oriented[key] = face
        owners.setdefault(key, int(body_index))
model_exterior_faces = [[] for _ in range(n_bodies)]
for key, count in counts.items():
    if count == 1:
        model_exterior_faces[owners[key]].append(vertices_xyz[list(oriented[key])])

# Boundary coordinates and lighting are prepared for both types of view.
azimuth, altitude = np.deg2rad([LIGHT_AZIMUTH, LIGHT_ALTITUDE])
light_direction = np.array(
    [
        np.cos(altitude) * np.cos(azimuth),
        np.cos(altitude) * np.sin(azimuth),
        np.sin(altitude),
    ]
)
local_body_faces, local_model_faces = [], []
body_face_colors, model_face_colors = [], []
for face_groups, local_groups, color_groups in (
    (body_faces, local_body_faces, body_face_colors),
    (model_exterior_faces, local_model_faces, model_face_colors),
):
    for body_index, faces in enumerate(face_groups):
        local_faces = np.asarray(faces, dtype=float).reshape(-1, 3, 3).copy()
        local_faces[..., 0] -= x_origin_m
        local_faces[..., 1] -= y_origin_m
        local_groups.append(local_faces)
        base = np.asarray(to_rgb(BODY_COLORS[body_index]), dtype=float)
        colors = []
        for face in faces:
            points = np.asarray(face, dtype=float)
            # Relief scaling applies only to the lighting calculation.
            display_points = points.copy()
            display_points[:, 0] = (points[:, 0] - x_origin_m) / x_span_m
            display_points[:, 1] = (points[:, 1] - y_origin_m) / y_span_m
            display_points[:, 2] = (
                (points[:, 2] - z_min_m)
                / (z_max_m - z_min_m)
                * Z_DISPLAY_SCALE
                * LIGHT_RELIEF_SCALE
            )
            normal = np.cross(
                display_points[1] - display_points[0],
                display_points[2] - display_points[0],
            )
            norm = np.linalg.norm(normal)
            diffuse = (
                abs(float(np.dot(normal / norm, light_direction))) if norm else 0.0
            )
            brightness = AMBIENT_LIGHT + DIFFUSE_LIGHT * diffuse
            colors.append((*np.clip(base * brightness, 0.0, 1.0), 1.0))
        color_groups.append(colors)

MODEL_X_TICKS = np.arange(0.0, x_span_m + 1e-9, 200.0) + x_origin_m
MODEL_Y_TICKS = np.arange(0.0, y_span_m + 1e-9, 200.0) + y_origin_m

## 4. Plot the full geological model


In [ ]:
with plt.rc_context(PLOT_RC):
    model_fig, model_ax = plt.subplots(
        figsize=(
            FULL_FIGURE_WIDTH_CM / CM_PER_INCH,
            FULL_FIGURE_HEIGHT_CM / CM_PER_INCH,
        ),
        subplot_kw={"projection": "3d"},
        dpi=600,
    )
    # The enlarged plotting area reduces unused vertical space.
    model_fig.subplots_adjust(left=0.02, right=0.98, bottom=-0.04, top=0.98)

    model_ax.view_init(elev=VIEW_ELEV, azim=VIEW_AZIM, roll=VIEW_ROLL)
    model_ax.set_proj_type("ortho")
    # The display box follows the model extent.
    model_ax.set_box_aspect(
        (x_span_m, y_span_m, Z_DISPLAY_SCALE * (z_max_m - z_min_m)), zoom=1.42
    )
    # A small margin prevents clipping of the last X-axis tick label.
    model_ax.set_xlim(0.0, x_span_m + 0.05 * x_span_m)
    model_ax.set_ylim(0.0, y_span_m)
    model_ax.set_zlim(z_min_m, z_max_m)

    faces_to_draw = [face for faces in local_model_faces for face in faces]
    colors_to_draw = [color for colors in model_face_colors for color in colors]
    model_ax.add_collection3d(
        Poly3DCollection(
            faces_to_draw,
            facecolors=colors_to_draw,
            edgecolors="none",
            linewidths=0,
        )
    )

    # A display-only offset prevents z-fighting between the Ground surface and mesh.
    ground_nodes = np.column_stack(
        (
            initial_nodes_xy,
            interfaces_z[:original_node_count, -1] + GROUND_MESH_OFFSET,
        )
    )
    ground_nodes[:, 0] -= x_origin_m
    ground_nodes[:, 1] -= y_origin_m
    mesh_segments = [
        ground_nodes[[*triangle, triangle[0]]] for triangle in initial_triangles
    ]
    model_ax.add_collection3d(
        Line3DCollection(
            mesh_segments,
            colors="#303944",
            linewidths=0.45,
            alpha=0.65,
        )
    )

    bx = np.array([row["x"] for row in completed_boreholes])
    by = np.array([row["y"] for row in completed_boreholes])
    bz = np.array([row["ground"] for row in completed_boreholes])
    model_ax.scatter(
        bx - x_origin_m,
        by - y_origin_m,
        bz + 1.5,
        s=12,
        c="#15191F",
        edgecolors="white",
        linewidths=0.6,
        depthshade=False,
        zorder=1,
    )
    for row in completed_boreholes:
        model_ax.text(
            row["x"] - x_origin_m,
            row["y"] - y_origin_m,
            row["ground"] + 3.0,
            row["drill"],
            fontsize=BOREHOLE_LABEL_SIZE,
            color="#1D2430",
            ha="center",
            va="bottom",
            zorder=5,
        )

    model_ax.set_xticks(MODEL_X_TICKS - x_origin_m)
    model_ax.set_xticklabels([f"{value:.0f}" for value in MODEL_X_TICKS])
    model_ax.set_yticks(MODEL_Y_TICKS - y_origin_m)
    model_ax.set_yticklabels([f"{value:.0f}" for value in MODEL_Y_TICKS])
    model_ax.zaxis.set_major_locator(MultipleLocator(Z_TICK_SPACING_M))
    # Elevation ticks are shown without an additional axis title.
    # Negative padding moves tick labels closer to the 3D axes.
    model_ax.tick_params(axis="both", which="major", labelsize=4, pad=-4)
    model_ax.grid(True)
    for axis in (model_ax.xaxis, model_ax.yaxis, model_ax.zaxis):
        # The 3D grid uses _axinfo settings and RGBA colours.
        axis._axinfo["grid"].update(
            color=to_rgba("lightgray", 0.8),
            linewidth=0.25,
            linestyle="-",
        )
        # _axinfo is a private API; rendering needs review after Matplotlib upgrades.
        axis._axinfo["tick"].update(inward_factor=0.0, outward_factor=0.2)
        axis._axinfo["tick"]["linewidth"][True] = 0.6

    model_ax.legend(
        handles=[
            Patch(facecolor=BODY_COLORS[body_index], label=BODY_LABELS[body_index])
            for body_index in reversed(range(n_bodies))
        ],
        # The upper-right margin provides space for the legend.
        loc="upper right",
        frameon=True,
        framealpha=0.9,
        borderpad=0.42,
        labelspacing=0.30,
        title="Geological units",
        fontsize=5,
        title_fontsize=7,
    )
    plt.savefig(FIGURE_DIR / "3D geological model.png", dpi=600)
    plt.show()

## 5. Plot individual geological units


In [ ]:
with plt.rc_context(PLOT_RC):
    unit_figures = {}
    unit_axes = []
    body_name_to_index = {name: index for index, name in enumerate(body_names)}
    missing_bodies = [
        name
        for name in [name for name, _, _ in UNIT_PANELS]
        if name not in body_name_to_index
    ]
    if missing_bodies:
        raise ValueError(
            f"The model is missing the selected geological units: {missing_bodies}"
        )
    selected_body_indices = [
        body_name_to_index[name] for name in [name for name, _, _ in UNIT_PANELS]
    ]
    for body_index, (_, _, key) in zip(selected_body_indices, UNIT_PANELS):
        unit_fig = plt.figure(
            figsize=(
                UNIT_FIGURE_WIDTH_CM / CM_PER_INCH,
                UNIT_FIGURE_HEIGHT_CM / CM_PER_INCH,
            ),
            dpi=600,
        )
        unit_ax = unit_fig.add_axes([0.01, 0.01, 0.98, 0.98], projection="3d")
        unit_ax.view_init(elev=VIEW_ELEV, azim=VIEW_AZIM, roll=VIEW_ROLL)
        unit_ax.set_proj_type("ortho")
        # The display box follows the model extent.
        unit_ax.set_box_aspect(
            (x_span_m, y_span_m, Z_DISPLAY_SCALE * (z_max_m - z_min_m)), zoom=1.42
        )
        # A small margin prevents clipping of the last X-axis tick label.
        unit_ax.set_xlim(0.0, x_span_m + 0.05 * x_span_m)
        unit_ax.set_ylim(0.0, y_span_m)
        unit_ax.set_zlim(z_min_m, z_max_m)
        unit_ax.set_xticks(MODEL_X_TICKS - x_origin_m)
        unit_ax.set_xticklabels([f"{value:.0f}" for value in MODEL_X_TICKS])
        unit_ax.set_yticks(MODEL_Y_TICKS - y_origin_m)
        unit_ax.set_yticklabels([f"{value:.0f}" for value in MODEL_Y_TICKS])
        unit_ax.zaxis.set_major_locator(MultipleLocator(Z_TICK_SPACING_M))
        unit_ax.set_xlabel("X (m)", labelpad=7)
        unit_ax.set_ylabel("Y (m)", labelpad=7)
        # Elevation ticks are shown without an additional axis title.
        unit_ax.set_zlabel("")
        unit_ax.tick_params(axis="both", which="major", labelsize=4, pad=-4)
        unit_ax.grid(True)
        for axis in (unit_ax.xaxis, unit_ax.yaxis, unit_ax.zaxis):
            # Individual-unit views share the full-model grid and tick styling.
            axis._axinfo["grid"].update(
                color=to_rgba("lightgray", 0.8),
                linewidth=0.15,
                linestyle="-",
            )
            axis._axinfo["tick"].update(inward_factor=0.0, outward_factor=0.2)
            axis._axinfo["tick"]["linewidth"][True] = 0.6
        # Individual-unit views share the model extent and viewpoint.
        # Axis text is hidden in these compact panels.
        unit_ax.set_box_aspect(
            (x_span_m, y_span_m, Z_DISPLAY_SCALE * (z_max_m - z_min_m)), zoom=1.12
        )
        unit_ax.set_xticklabels([])
        unit_ax.set_yticklabels([])
        unit_ax.set_zticklabels([])
        unit_ax.set_xlabel("")
        unit_ax.set_ylabel("")
        unit_ax.add_collection3d(
            Poly3DCollection(
                local_body_faces[body_index],
                facecolors=body_face_colors[body_index],
                edgecolors="none",
                linewidths=0,
            )
        )
        output_dir = FIGURE_DIR
        output_dir.mkdir(parents=True, exist_ok=True)

        unit_fig.savefig(
            output_dir / f"{key}.png",
            dpi=600,
            facecolor="white",
        )
        unit_figures[key] = unit_fig
        unit_axes.append(unit_ax)
    plt.show()